In [1]:
import pathlib
from argparse import ArgumentParser
import yaml

from pytorch_lightning import Trainer
from pytorch_lightning.callbacks import ModelCheckpoint

In [2]:
import sys 
sys.path.append('../')
from src.coch_word_rec_lightning import CochWordRecModule


In [3]:
path = '../config/coch_word_rec/word_rec_1e-4lr.yaml'
config = yaml.load(open(path, 'r'), Loader=yaml.FullLoader)

In [4]:
config

{'data': {'num_words': 794,
  'corpus': {'with_noise': True,
   'root': '/om4/group/mcdermott/projects/ibmHearingAid/assets/data/datasets/JSIN_v3.00/nStim_20000/2000ms/rms_0.1/noiseSNR_-10_10/stimSR_20000/reverb_none/noise_all/JSIN_all_v3/subsets'},
  'audio': {'rep_type': 'cochlea_filt',
   'rep_kwargs': {'sr': 20000,
    'env_sr': 8000,
    'n_channels': 40,
    'low_lim': 40,
    'use_pad': True,
    'rep_on_gpu': False,
    'env_extraction_type': 'Half-wave Rectification',
    'downsampling_type': 'TorchTransformsResample',
    'downsampling_kwargs': {'lowpass_filter_width': 64,
     'rolloff': 0.9475937167399596,
     'resampling_method': 'kaiser_window',
     'beta': 14.769656459379492}},
   'compression_type': 'coch_p3',
   'compression_kwargs': {'scale': 1,
    'offset': 1e-07,
    'clip_value': 5,
    'power': 0.3}},
  'loader': {'batch_size': 16}},
 'val_metric': 'ACC/val_acc',
 'model_name': 'CochCNN',
 'hparas': {'valid_step': 5000,
  'epochs': 6,
  'optimizer': 'Adam',
  '

In [5]:
config['n_jobs'] = 10
config['data']['loader']['batch_size'] = 1

In [6]:
model = CochWordRecModule(config)

In [11]:
trainer = Trainer(
#     default_root_dir='../test_log_dump/',
   # log_every_n_steps = 10,
#     limit_train_batches=0.1,
    limit_test_batches=0.1,

    num_nodes=1,
    gpus=1,
    accelerator="gpu",
)

GPU available: True, used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs


In [12]:
ckpt_path = '../word_rec/jsin_precombined_gammatone_40_channels_20kHz_1e-4lr/checkpoints/epoch=0-step=39999.ckpt'
trainer.test(model, ckpt_path=ckpt_path)

Restoring states from the checkpoint path at ../word_rec/jsin_precombined_gammatone_40_channels_20kHz_1e-4lr/checkpoints/epoch=0-step=39999.ckpt
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Loaded model weights from checkpoint at ../word_rec/jsin_precombined_gammatone_40_channels_20kHz_1e-4lr/checkpoints/epoch=0-step=39999.ckpt


Testing: 0it [00:00, ?it/s]

--------------------------------------------------------------------------------
DATALOADER:0 TEST RESULTS
{'ACC/test_acc': 0.3491969108581543,
 'ACC/test_acc_epoch': 0.3491969108581543,
 'Losses/test_loss': 3.384685516357422,
 'Losses/test_loss_epoch': 3.384685516357422}
--------------------------------------------------------------------------------


[{'Losses/test_loss': 3.384685516357422,
  'Losses/test_loss_epoch': 3.384685516357422,
  'ACC/test_acc': 0.3491969108581543,
  'ACC/test_acc_epoch': 0.3491969108581543}]